## Complete Dictionary with included events as desired - May be exported for one time completion 

In [1]:
import os
import re
import pathlib as pl
import collections

In [2]:
os.chdir('..')
home = pl.Path(os.getcwd())

#user to set variables for the project. tagged as parameter for papermill runs
project = 'wy_fy23'

In [3]:
print('home is at: ',home)
home = pl.Path(home)
from src.hdf import *

inputs = home/'inputs'
outputs_base = home/'outputs'

export_folder = outputs_base/project/'event_tie_ins'

home is at:  c:\_code\hms_to_ras_sst


In [4]:
#create folder to export the tie-in dictionaries.
if not os.path.exists(export_folder):
    os.makedirs(export_folder)

In [5]:
with open(inputs/project/'dictionaries'/'HUC10_outflow_toHUC10.json') as src:
    huc_connect_huc = json.load(src)
with open(inputs/project/'dictionaries'/'HUC10_into_dsJunction.json') as src:
    huc_connect_j = json.load(src)

In [7]:
#list of all applicable HUC10s
huc_connect_huc.keys()

complete_dictionary_p1 = {}

for huc in huc_connect_huc.keys():
    prob_shps = glob.glob(str(inputs/project/'hms_ras'/huc/'*all_junctions_events.geojson'))
    complete_dictionary_p1.update(get_sst_storms_by_recurrence(huc,prob_shps))

IndexError: list index out of range

In [ ]:
#create separate temporary dictionary to add to the list shown in the events
completed_us_dictionary_p2 = {}

for huc in huc_connect_huc.keys():
    us_huc_temp = [k for k,v in huc_connect_huc.items() if v == huc]
    if len(us_huc_temp) == 0:
        print(f'huc {huc} is most upstream')
    else:
        bc_connections = {'junctions':{},'dss_path':{},'ts':{}}
        
        us_hucs = [k for k in us_huc_temp]
        ds_j = [huc_connect_j[k] for k in us_huc_temp]
        
        for us in us_hucs:
            if us not in ds_junc_adjustments.keys():
                bc_connections['junctions'][us] = huc_connect_j[us]
                bc_connections['dss_path'][us] = {}
            else:
                bc_connections['junctions'][us] = ds_junc_adjustments[us]
                bc_connections['dss_path'][us] = {}
                
        events_dict_us = {}
        print(f'filling out dictionary for huc {huc}')
        for huc, j in  bc_connections['junctions'].items():
            events_dict_us[huc] = {j:{}}
            htmls = glob.glob(str(inputs/project/'hms_ras'/huc/'plots'/'*_map.html'))
            events_dict_us = get_sst_storms_by_recurrence_us_huc(events_dict_us,huc,htmls,j)
            
        completed_us_dictionary_p2.update(events_dict_us)

len(completed_us_dictionary_p2)

huc 1404010102 is most upstream
huc 1404010103 is most upstream
huc 1404010104 is most upstream
huc 1404010105 is most upstream
huc 1404010106 is most upstream
filling out dictionary for huc 1404010107
huc 1404010108 is most upstream
filling out dictionary for huc 1404010109
huc 1404010110 is most upstream
filling out dictionary for huc 1404010111
huc 1404010112 is most upstream
filling out dictionary for huc 1404010113
huc 1404010201 is most upstream
huc 1404010202 is most upstream
huc 1404010203 is most upstream
huc 1404010204 is most upstream
huc 1404010205 is most upstream
filling out dictionary for huc 1404010206
huc 1404010301 is most upstream
huc 1404010302 is most upstream
filling out dictionary for huc 1404010303
huc 1404010304 is most upstream
huc 1404010305 is most upstream
filling out dictionary for huc 1404010306
huc 1404010401 is most upstream
huc 1404010402 is most upstream
filling out dictionary for huc 1404010403
huc 1404010404 is most upstream
filling out dictionary f

59

In [ ]:
import json
#export master combined list to json. should only be done once cannot be redone or may overwrite previous list.
with open(export_folder/'completed_event_dictionary.json', "w") as outfile:
    json.dump(complete_dictionary_p1, outfile, indent= 1)

In [ ]:
#WARNING --- This dictionary has had the downstream junctions modified for a select number of HUCs in order to attain the events related to it. Be advised they will not match ds junctions 100%

import json
#export master combined list to json. should only be done once cannot be redone or may overwrite previous list.
with open(export_folder/'tie_in_dictionary.json', "w") as outfile2:
    json.dump(completed_us_dictionary_p2, outfile2, indent= 1)

Trial Checking max at the downstream junctions

In [ ]:
#create a junction class that holds the relevant info for each junction


In [70]:
huc_list = ['1008001201','1008001202','1008001301','1008001302','1008001303']

#dictionary to hold all recurrence interval, event and flow information together for all junctions.
ALL_model_recur_event_flows = {}

for huc in huc_list:
    geojson_paths = glob.glob(str(inputs/project/'hms_ras'/huc/'*all_junctions_events.geojson'))


    #attain all recurrence interval information
    for path in geojson_paths:
        basename = pl.Path(path).stem
        recur_find = re.search(r'_[\d.]+[_mp]', basename)
        recur = recur_find.group(0).replace('_','')

        #identify all events related to this recurrence interval
        with open(path, 'r') as f:
            geojson_data = json.load(f)
        key_list = list(geojson_data['features'][0]['properties'].keys())
        event_matches = [item for item in key_list if re.search(r'P[\d]+_R[EY\-\d]+', item, re.IGNORECASE)]
        
        #iterate through each junction to collect information related to it.
        for i in range(0,len(geojson_data['features'])):
            j_name = (geojson_data['features'][i]['properties']['name'])
            j_events = [key for key in list(geojson_data['features'][0]['properties'].keys()) if key in event_matches]
            assert len(j_events) == len(event_matches), "Mismatch in number of events found"
            
            #in addition to flow per event at junction, the max flow event is also identified and stored.
            temp_event_flow = {}
            for event in j_events:
                ALL_model_recur_event_flows.setdefault(huc, {}).setdefault(j_name, {}).setdefault(recur, {}).setdefault('events', {})[event] = geojson_data['features'][i]['properties'][event]
                temp_event_flow.update({event: geojson_data['features'][i]['properties'][event]})
            max_flow_per_recur = max(temp_event_flow, key=temp_event_flow.get)
            ALL_model_recur_event_flows[huc][j_name][recur].update({'max_flow_event':{max_flow_per_recur:geojson_data['features'][i]['properties'][max_flow_per_recur]}})

with open(export_folder/"all_model_event_data.json", "w") as f:
    json.dump(ALL_model_recur_event_flows, f, indent=4)
        
        

In [74]:
huc_connect_huc['1008001303'] = 'OUT'
huc_connect_huc['1008001202'] = 'OUT'

all_model_ds_data = {}

for huc in huc_list:
    if huc_connect_huc[huc] != 'OUT' and huc_connect_j[huc] != 'N\A':
        dsj = huc_connect_j[huc]
        dshuc = huc_connect_huc[huc]

        for recur_int in ALL_model_recur_event_flows[huc][dsj].keys():
            tie_in_event = list(ALL_model_recur_event_flows[huc][dsj][recur_int]['max_flow_event'].keys())[0]
            tie_in_event_flow = list(ALL_model_recur_event_flows[huc][dsj][recur_int]['max_flow_event'].values())[0]

            upper_limit_event = list(ALL_model_recur_event_flows[dshuc][dsj][recur_int]['max_flow_event'].keys())[0]
            upper_limit_flow = list(ALL_model_recur_event_flows[dshuc][dsj][recur_int]['max_flow_event'].values())[0]

            us_ds_flow_ratio = round(float(tie_in_event_flow)/float(upper_limit_flow),3)

            events_exceed = {k:v for k,v in ALL_model_recur_event_flows[huc][dsj][recur_int]['events'].items() if v > upper_limit_flow and k != tie_in_event}
            if events_exceed:
                max_event_events_exceeded = max(events_exceed, key=events_exceed.get)
                max_flow_events_exceeded = max(events_exceed.values())
                max_flow_ratio_to_dshuc = round(float(max_flow_events_exceeded)/float(upper_limit_flow),3)
            else:
                max_event_events_exceeded = None
                max_flow_events_exceeded = None
                max_flow_ratio_to_dshuc = None
            all_model_ds_data.setdefault(huc, {}).setdefault(dsj, {}).setdefault(recur_int, {}).update({'ds_huc': dshuc,
                                                                                                        'us_tie_in_event': tie_in_event,
                                                                                                        'us_tie_in_flow': tie_in_event_flow,
                                                                                                        'ds_tie_in_event': upper_limit_event,
                                                                                                        'ds_tie_in_flow': upper_limit_flow,
                                                                                                        'us_ds_flow_ratio': us_ds_flow_ratio,
                                                                                                        'us_ds_flow_ratio_flagged': True if us_ds_flow_ratio >= 1.2 or us_ds_flow_ratio <= 0.8 else False,
                                                                                                        'us_events_exceeding_ds_tie_in': events_exceed,
                                                                                                        'max_event_from_exceeding': max_event_events_exceeded,
                                                                                                        'max_flow_from_exceeding': max_flow_events_exceeded,
                                                                                                        'max_flow_ratio_us_ds': max_flow_ratio_to_dshuc,
                                                                                                        'max_flow_ratio_flagged': True if max_flow_ratio_to_dshuc != None and max_flow_ratio_to_dshuc >= 1.2 else False})
with open(export_folder/"all_model_ds_event_data.json", "w") as f:
    json.dump(all_model_ds_data, f, indent=4)

In [69]:
[k for k,v in huc_connect_j.items() if v == 'HUC_014_J_359']

['1008001203']

In [45]:
all_model_ds_data

{'1008001301': {'HUC_013_J_33': {'0.002': {'ds_huc': '1008001302',
    'dsj_upper_limit_flow': 7947.419921875,
    'exceeding_events': {'P07_R-Y035-E0001': 8031.259765625,
     'P01_R-Y465-E0002': 8877.400390625,
     'P01_R-Y048-E0001': 8823.6396484375,
     'P05_R-Y198-E0002': 8033.0712890625},
    'max_flow_from_exceeding_events': 8877.400390625,
    'max_flow_ratio_us_dshuc': 1.117},
   '0.01m': {'ds_huc': '1008001302',
    'dsj_upper_limit_flow': 7269.2514648438,
    'exceeding_events': {'P09_R-Y300-E0003': 7305.0400390625},
    'max_flow_from_exceeding_events': 7305.0400390625,
    'max_flow_ratio_us_dshuc': 1.005},
   '0.01p': {'ds_huc': '1008001302',
    'dsj_upper_limit_flow': 8239.0400390625,
    'exceeding_events': {'P07_R-Y019-E0002': 8365.0302734375,
     'P01_R-Y465-E0002': 8877.400390625,
     'P07_R-Y357-E0003': 8299.919921875},
    'max_flow_from_exceeding_events': 8877.400390625,
    'max_flow_ratio_us_dshuc': 1.077},
   '0.01': {'ds_huc': '1008001302',
    'dsj_upper

In [44]:

########next to do is to identify the junctions that overlap each plot and determine if any of the other events in the upstream is much greater than the downstream.